In [1]:


from pipeline_utils.ClassificationHeads import BaselineClassificationHead
from Transformer import *
from transformers import AutoTokenizer
import math
import re
from dataset_utils import *
from pipeline_utils.pipeline_utils import create_profile_vector
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
device = "cuda"

/home/max/ProgrammingProjects/-Social-Media-Bot-Detection-with-Continuous-Learning/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class DefaultEncoder(nn.Module):
    def __init__(self,src_vocab_size,embed_size,num_layers,heads,device,forward_expansion,dropout,max_length, activation_function = nn.ReLU()):
        super(DefaultEncoder, self).__init__()
        self.encoder = Encoder(src_vocab_size,embed_size,num_layers,heads,device,forward_expansion,dropout, max_length, activation_function)

    def forward(self,x , msk):
        encoder_out = self.encoder(x, msk)
        return encoder_out

class ModifiedEncoder(nn.Module):
    def __init__(self,src_vocab_size,embed_size,num_layers,heads,device,forward_expansion,dropout,max_length, activation_function = nn.ReLU()):
        super(ModifiedEncoder, self).__init__()
        self.encoder = EncoderModified(src_vocab_size,embed_size,num_layers,heads,device,forward_expansion,dropout, max_length, activation_function)

    def forward(self,x , msk):
        encoder_out = self.encoder(x, msk)
        return encoder_out


In [3]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

VOCAB_SIZE = tokenizer.vocab_size
EMBED_SIZE = 128
NUM_LAYERS = 3
HEADS = 8
FORWARD_EXPANSION = 4
DROPOUT = 0.1
MAX_LENGTH = 768


def create_tweet_vectors(tweet_data, tokenizer, model, batch_size = math.inf, max_tweets = math.inf, device ='cuda' if torch.cuda.is_available() else 'cpu'):
    """
    creates the tweets embedding vector for the provided user.
    :param device: the device on which to process the data.
    :param model: the model used for creating semantic embeddings from tweets
    :param tokenizer: the tokenizer used for creating tokens from text strings
    :param tweet_data: a dictionary corresponding to one sample from the dataset. Should contain a 'tweets' entry.
    :param batch_size: the batch size used to process the tweets. Defaults to processing all tweets at once
    :param max_tweets: the maximum amount of tweets to process per profile. Defaults to processing all available tweets at once.
    :return: tensor of the tweets embedding vector
    """
    tweet_count = len(tweet_data)
    # if no tweets are available, return a default vector
    if tweet_count == 0:
        return torch.atleast_2d(torch.zeros(768,1)).to(device)

    # get the maximum wanted tweets to process
    texts = list(map(lambda x: x.text, tweet_data))[0: min(tweet_count, max_tweets)]
    # process texts in batches
    raw_embeddings = []
    index = 0
    # replace urls with 'url'
    URL_PATTERN = r"https?://\S+|www\.\S+"
    texts = [re.sub(URL_PATTERN, " url ", text) for text in texts]
    while index < min(tweet_count, max_tweets):
        # grab text batch
        text_batch = texts[index:min(index+batch_size, tweet_count)]

        # get textual embedding of the batch
        tokens = tokenizer(text_batch, padding="max_length", truncation=True, max_length=MAX_LENGTH,return_tensors="pt").to(device)
        input_ids = tokens["input_ids"].to(device)
        mask = tokens["attention_mask"].unsqueeze(1).unsqueeze(2).to(device)
        output = model(input_ids, mask)
        # collect embeddings
        raw_embeddings.extend(torch.unbind(output,dim=0))
        final_embeddings = torch.stack(raw_embeddings, dim=0)

        index += batch_size
    final_embeddings = torch.mean(final_embeddings, dim=0)
    return torch.atleast_2d(final_embeddings)

def evaluation_loop(encoder_model, classifier, train_dataset, test_dataset):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(list(classifier.parameters()) + list(encoder_model.parameters()), lr=0.01)
    encoder_model.train()
    classifier.train()

    # training loop
    ground_truth_labels = []
    predictions = []
    for i, sample in enumerate(train_dataset):
        # get feature vectors
        profile_embed = create_profile_vector(sample.user_data).to(device)
        tweet_embeds = create_tweet_vectors(sample.tweet_data, tokenizer, encoder_model, max_tweets= 50, batch_size = 50)
        # pool tweet vectors
        tweet_vec = torch.mean(tweet_embeds, dim=1, dtype=torch.float32)
        # classify
        embeds = torch.concat([tweet_vec, profile_embed], dim=0).to(device)
        outputs = classifier(embeds)

        loss = criterion(outputs, torch.tensor(sample.label).to(device))
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        ground_truth_labels.append(sample.label)
        predictions.append(torch.argmax(outputs).cpu().detach().item())

        print(f"\rIteration: {i}", end="")
    print("accuracy: ", accuracy_score(ground_truth_labels, predictions))
    print("recall: ", recall_score(ground_truth_labels, predictions, average="macro"))
    print("f1 score: ", f1_score(ground_truth_labels, predictions, average="macro"))
    print(confusion_matrix(ground_truth_labels, predictions))

    # test loop
    encoder_model.eval()
    classifier.eval()
    ground_truth_labels = []
    predictions = []
    for i, sample in enumerate(test_dataset):
        # get feature vectors
        profile_embed = create_profile_vector(sample.user_data).to(device)
        with torch.no_grad():
            tweet_embeds = create_tweet_vectors(sample.tweet_data, tokenizer, encoder_model, max_tweets= 50, batch_size = 50)

            # pool tweet vectors
            tweet_vec = torch.mean(tweet_embeds, dim=1, dtype=torch.float32)
            # classify
            embeds = torch.concat([tweet_vec, profile_embed], dim=0).to(device)
            outputs = classifier(embeds)

            ground_truth_labels.append(sample.label)
            predictions.append(torch.argmax(outputs).cpu().detach().item())

        print(f"\rIteration: {i}", end="")
    print("accuracy: ", accuracy_score(ground_truth_labels, predictions))
    print("recall: ", recall_score(ground_truth_labels, predictions, average="macro"))
    print("f1 score: ", f1_score(ground_truth_labels, predictions, average="macro"))
    print(confusion_matrix(ground_truth_labels, predictions))


In [4]:
train_dataset = InterleavedIterableDataset([
            Cresci17(Cresci17SetTypes.GENUINE_USER, "train", root="../datasets", custom_label=0),
            Cresci17(Cresci17SetTypes.FAKE_FOLLOWER, "train", root="../datasets", custom_label=1),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_1, "train", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_2, "train", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_3, "train", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_1, "train", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_2, "train", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_3, "train", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_4, "train", root="../datasets", custom_label=3),
        ], "Random")
test_dataset = InterleavedIterableDataset([
            Cresci17(Cresci17SetTypes.GENUINE_USER, "test", root="../datasets", custom_label=0),
            Cresci17(Cresci17SetTypes.FAKE_FOLLOWER, "test", root="../datasets", custom_label=1),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_1, "test", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_2, "test", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_3, "test", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_1, "test", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_2, "test", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_3, "test", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_4, "test", root="../datasets", custom_label=3),
        ], "Random")

In [5]:
# default
classifier = BaselineClassificationHead(775,4,0.1).to(device)
encoder_model = DefaultEncoder(VOCAB_SIZE, EMBED_SIZE, NUM_LAYERS, HEADS, "cuda", FORWARD_EXPANSION, DROPOUT, MAX_LENGTH).to("cuda")
evaluation_loop(encoder_model, classifier, train_dataset, test_dataset)

Iteration: 10144accuracy:  0.6862493839329719
recall:  0.6882487303641533
f1 score:  0.6924834519330978
[[2317   37   89  120]
 [  14 1324  795  215]
 [  30 1018 1935  210]
 [  56  173  426 1386]]
Iteration: 1310accuracy:  0.4889397406559878
recall:  0.4109907120743034
f1 score:  0.33490436444355837
[[208   0 115   0]
 [  0   0 281   0]
 [  0   0 433   0]
 [  7   0 267   0]]


In [6]:
# other activation function
classifier = BaselineClassificationHead(775,4,0.1).to(device)
encoder_model = DefaultEncoder(VOCAB_SIZE, EMBED_SIZE, NUM_LAYERS, HEADS, "cuda", FORWARD_EXPANSION, DROPOUT, MAX_LENGTH, activation_function=nn.SiLU()).to("cuda")
evaluation_loop(encoder_model, classifier, train_dataset, test_dataset)

Iteration: 10144accuracy:  0.6042385411532775
recall:  0.6001112660726236
f1 score:  0.6144123402398018
[[1676  230  520  137]
 [   7  883 1231  227]
 [   4  846 2145  198]
 [  39   74  502 1426]]
Iteration: 1310accuracy:  0.4958047292143402
recall:  0.4179566563467492
f1 score:  0.31255690250807544
[[217   0 106   0]
 [  4   0 277   0]
 [  0   0 433   0]
 [164   0 110   0]]


In [5]:
# added normalization before attention
classifier = BaselineClassificationHead(775,4,0.1).to(device)
encoder_model = ModifiedEncoder(VOCAB_SIZE, EMBED_SIZE, NUM_LAYERS, HEADS, "cuda", FORWARD_EXPANSION, DROPOUT, MAX_LENGTH).to("cuda")
evaluation_loop(encoder_model, classifier, train_dataset, test_dataset)

Iteration: 10144accuracy:  0.6137999014292755
recall:  0.6119979242151631
f1 score:  0.6319141813558941
[[1892  245  340   86]
 [  24 1145 1075  104]
 [  24 1124 1928  117]
 [  56  214  509 1262]]
Iteration: 1310accuracy:  0.4950419527078566
recall:  0.4171826625386997
f1 score:  0.33727335417979476
[[216   0 107   0]
 [  0   0 281   0]
 [  0   0 433   0]
 [ 18   0 256   0]]
